In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime, timedelta
import lightgbm as lgb
import joblib
import matplotlib.cm as cm
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# META DATAPROCESS

In [ ]:
meta_file = "data/metadata.csv"
df_meta = pd.read_csv(meta_file)

# feature_selected # check trong file metadata_process.ipynb
combo_features1 = ["electricity", "chilledwater", "hotwater"]

df_meta_selected = df_meta[df_meta[combo_features1].eq("Yes").all(axis=1)].reset_index()
# print(df_meta.columns)
# filter features
dataset_features = ['building_id', 'site_id','primaryspaceusage', 'sqm', 'sqft']
df_meta_selected = df_meta_selected[dataset_features]
print(df_meta_selected.isna().sum())
BUILDIING_SELECTED = df_meta_selected['building_id'].to_list()
print(BUILDIING_SELECTED)
df_meta_selected

In [ ]:
file = "data/electricity_cleaned.csv"
df_e = pd.read_csv(file)

# Filter data by building list (electricity_cleaned, chilledwater_cleaned, hot_water_cleaned)

In [ ]:
threshold = 400 # nan
print("Threshold: ", threshold)  # số giá trị bị nan cả từng tòa nhà


file = "data/electricity_cleaned.csv"
df_e = pd.read_csv(file)
print(len(df_e))
# df_e.head()
df_e_selected = df_e[BUILDIING_SELECTED]
df_e_selected.head()
nan_count_e = df_e_selected.isna().sum().sort_values()
print("len BUILDIING_SELECTED: ", len(BUILDIING_SELECTED))
BUILDIING_SELECTED = nan_count_e[nan_count_e < threshold].index.tolist()
nan_count_e = nan_count_e[nan_count_e < threshold]
print("electricity => BUILDIING_SELECTED: ", len(BUILDIING_SELECTED))



file = "data/chilledwater_cleaned.csv"
df_chille_water = pd.read_csv(file)
print(len(df_chille_water))
# df_chille_water.head()
df_chille_water_selected = df_chille_water[BUILDIING_SELECTED]
df_chille_water_selected.head()
nan_count_chille_water = df_chille_water_selected.isna().sum().sort_values()
BUILDIING_SELECTED = nan_count_chille_water[nan_count_chille_water < threshold].index.tolist()
nan_count_chille_water = nan_count_chille_water[nan_count_chille_water < threshold]
print("chille => BUILDIING_SELECTED: ", len(BUILDIING_SELECTED))



file = "data/hotwater_cleaned.csv"
df_hot_water = pd.read_csv(file)
print(len(df_hot_water))
# hot_water.head()
hot_water_selected = df_hot_water[BUILDIING_SELECTED]
hot_water_selected.head()
nan_count_hot_water = hot_water_selected.isna().sum().sort_values()
BUILDIING_SELECTED = nan_count_hot_water[nan_count_hot_water < threshold].index.tolist()
nan_count_hot_water = nan_count_hot_water[nan_count_hot_water < threshold]
print("hot_water => BUILDIING_SELECTED: ", len(BUILDIING_SELECTED))

print("BUILDIING_SELECTED: ", len(BUILDIING_SELECTED))


In [ ]:
buildings = nan_count_e.index

nan_count_chille_water = nan_count_chille_water.reindex(buildings)
nan_count_hot_water = nan_count_hot_water.reindex(buildings)
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(buildings, nan_count_e.values, label="Electricity NaN")
plt.plot(buildings, nan_count_chille_water.values, label="Chilled Water NaN")
plt.plot(buildings, nan_count_hot_water.values, label="Hot NaN")

plt.xlabel("Building")
plt.ylabel("NaN count")
plt.title("NaN count per building")
plt.legend()
plt.xticks(rotation=90)
plt.grid(True)

plt.tight_layout()
plt.show()


# UPDATA data with BUILDIING_SELECTED

In [ ]:
data_metadata_indexed = df_meta_selected[df_meta_selected["building_id"].isin(BUILDIING_SELECTED)].set_index('building_id')

df_e_selected = df_e[["timestamp"] + BUILDIING_SELECTED]
df_e_selected['timestamp'] = pd.to_datetime(df_e_selected['timestamp'])
df_e_selected.set_index('timestamp', inplace=True)

df_chille_water_selected = df_chille_water[["timestamp"] + BUILDIING_SELECTED]
df_chille_water_selected['timestamp'] = pd.to_datetime(df_chille_water_selected['timestamp'])
df_chille_water_selected.set_index('timestamp', inplace=True)

df_hot_water_selected = df_hot_water[["timestamp"] + BUILDIING_SELECTED]
df_hot_water_selected['timestamp'] = pd.to_datetime(df_hot_water_selected['timestamp'])
df_hot_water_selected.set_index('timestamp', inplace=True)


# Weather

In [ ]:
df_weather = pd.read_csv('data/weather.csv')   # data giờ quét nhiều giá trị
df_weather["timestamp"] = pd.to_datetime(df_weather["timestamp"])
print(len(df_weather))
print(df_weather.columns)
# ktra so luong Nan ở các cột
count_na = df_weather.isna().sum()
# count_na
# df_w = df_w[['timestamp', 'airTemperature', 'dewTemperature', 'windSpeed']].dropna()
df_weather = df_weather[['timestamp','airTemperature', 'dewTemperature', 'windSpeed']].copy().set_index('timestamp')
df_weather.sort_values("timestamp")
df_weather.index.duplicated().sum()
df_weather_clean = (
    df_weather
    .groupby(level=0)
    .mean()      
)
print(len(df_weather_clean))
df_weather_clean

In [ ]:
# feature columns
X_cols = [
    "hour","dayofweek","is_weekend","month",
    "lag_24","rolling_24",
    "airTemperature", "dewTemperature", "windSpeed",  # weather
    "temp_lag_1h","dewTemperature_lag_1h", "windSpeed_lag_1h",
    "sqft", "sqm", 
    "primaryspaceusage", "site_id", "building_id",
    'chilled_delta', 'hot_delta',
    "Chilledwater", "Hotwater",
    'chilled_per_sqm', 'hot_per_sqm',
    'chilled_ratio_e', 'hot_ratio_e',
    'chilled_ratio_e_lag1', 'hot_ratio_e_lag1',
    'thermal_balance',
    'thermal_load', 'thermal_load_per_sqm'
]

# PREPARE DATA FOR SITE: Cockatoo

In [ ]:
# df_electric = df_model[['Cockatoo_lodging_Aimee']]
# df_merged = df_electric.join(df_weather_clean, how='left')
# df_merged.head()
# df_weather_clean

In [ ]:
import os
import pandas as pd

save_data = "data_1578_csv"
os.makedirs(save_data, exist_ok=True)

list_building = [f for f in os.listdir(save_data)]
forecast_horizon = 24

for name_building in df_e_selected.columns:
    if name_building in list_building:
        print(f"{name_building} already processed, skipping...")
        continue

    try:
        site = name_building.split('_')[0]
        print(f"Processing building: {name_building} (site: {site})")

        # =========================
        # 1. LẤY ĐIỆN THEO BUILDING
        # =========================
        if name_building not in df_e_selected:
            print(f"No data for {name_building}, skipping...")
            continue

        df_electric_b_x = df_e_selected[[name_building]].rename(columns={name_building: "Electricity"})
        df_chilledwater = df_chille_water_selected[[name_building]].rename(columns={name_building: "Chilledwater"})
        df_hot_water = df_hot_water_selected[[name_building]].rename(columns={name_building: "Hotwater"})

        if df_electric_b_x is None or df_electric_b_x.empty:
            print(f"DataFrame is None or empty for {name_building}, skipping...")
            continue

        # =========================
        # 2. MERGE THỜI TIẾT
        # =========================
        df_train_test_x = df_electric_b_x.join(df_weather_clean, how='left')
        df_train_test_x = df_train_test_x.join(df_chilledwater, how="left")
        df_train_test_x = df_train_test_x.join(df_hot_water, how="left")
        
        if df_train_test_x.empty:
            print(f"Merged DataFrame is empty for {name_building}, skipping...")
            continue

        df_train_test_x.sort_index(inplace=True)

        # =========================
        # 3. FEATURE ENGINEERING
        # =========================
        df_train_test_x["hour"] = df_train_test_x.index.hour
        df_train_test_x["dayofweek"] = df_train_test_x.index.dayofweek
        df_train_test_x["is_weekend"] = (df_train_test_x["dayofweek"] >= 5).astype(int)
        df_train_test_x["month"] = df_train_test_x.index.month

        df_train_test_x["lag_24"] = df_train_test_x["Electricity"].shift(24)
        df_train_test_x["rolling_24"] = df_train_test_x["Electricity"].rolling(24).mean()

        df_train_test_x["temp_lag_1h"] = df_train_test_x["airTemperature"].shift(1)
        df_train_test_x["dewTemperature_lag_1h"] = df_train_test_x["dewTemperature"].shift(1)
        df_train_test_x["windSpeed_lag_1h"] = df_train_test_x["windSpeed"].shift(1)

        
        # =========================
        # 4. METADATA BUILDING
        # =========================
        if name_building not in data_metadata_indexed.index:
            print(f"No metadata for {name_building}, skipping...")
            continue

        meta = data_metadata_indexed.loc[name_building]

        df_train_test_x["sqft"] = meta.get("sqft", 0)
        df_train_test_x["sqm"] = meta.get("sqm", 0)
        
        df_train_test_x["primaryspaceusage"] = meta.get("primaryspaceusage", "unknown")
        df_train_test_x["site_id"] = meta.get("site_id", "unknown")
        df_train_test_x["building_id"] = name_building
        
        df_train_test_x["chilled_delta"] = df_train_test_x["Chilledwater"] - df_train_test_x["Chilledwater"].shift(1)
        df_train_test_x["hot_delta"] = df_train_test_x["Hotwater"] - df_train_test_x["Hotwater"].shift(1)
        
        df_train_test_x["chilled_per_sqm"] = df_train_test_x["Chilledwater"] / df_train_test_x["sqm"]
        df_train_test_x["hot_per_sqm"] = df_train_test_x["Hotwater"] / df_train_test_x["sqm"]
        
        df_train_test_x["chilled_ratio_e"] = df_train_test_x["Chilledwater"] / df_train_test_x["Electricity"]
        df_train_test_x["hot_ratio_e"] = df_train_test_x["Hotwater"] / df_train_test_x["Electricity"]
        
        df_train_test_x["chilled_ratio_e_lag1"] = (
            df_train_test_x["Chilledwater"].shift(1) /
            df_train_test_x["Electricity"].shift(1)
        )

        df_train_test_x["hot_ratio_e_lag1"] = (
            df_train_test_x["Hotwater"].shift(1) /
            df_train_test_x["Electricity"].shift(1)
        )

        df_train_test_x["thermal_balance"] = (
            df_train_test_x["Chilledwater"] - df_train_test_x["Hotwater"]
        ) / (
            df_train_test_x["Chilledwater"] + df_train_test_x["Hotwater"] + 1e-6
        )*100
        
        df_train_test_x["thermal_load"] = (
            df_train_test_x["Chilledwater"] + df_train_test_x["Hotwater"]
        )

        df_train_test_x["thermal_load_per_sqm"] = (
            df_train_test_x["thermal_load"] / (df_train_test_x["sqm"] + 1e-6)
        )

        # =========================
        # 5. DROP NA (SAU KHI TẠO FEATURE)
        # =========================
        df_train_test_x.dropna(inplace=True)
        if df_train_test_x.empty:
            print(f"All rows dropped due to NA for {name_building}, skipping...")
            continue

        # =========================
        # 6. TRAIN / TEST SPLIT
        # =========================
        cutoff = (df_train_test_x.index.max() - pd.DateOffset(months=4)).replace(hour=0)
        train_df_x = df_train_test_x[df_train_test_x.index < cutoff].copy()
        test_df_x  = df_train_test_x[df_train_test_x.index >= cutoff].copy()

        if train_df_x.empty:
            print(f"Train set is empty for {name_building}, skipping...")
            continue

        # =========================
        # 7. MULTI-HORIZON TARGET
        # =========================
        for h in range(forecast_horizon):
            col = f"target_t+{h+1}"
            train_df_x[col] = train_df_x["Electricity"].shift(-(h+1))
        train_df_x.dropna(inplace=True)

        if train_df_x.empty:
            print(f"Train set empty after shifting target for {name_building}, skipping...")
            continue

        # =========================
        # 8. SAVE CSV
        # =========================
        building_dir = os.path.join(save_data, name_building)
        os.makedirs(building_dir, exist_ok=True)

        train_df_x.to_csv(os.path.join(building_dir, "train.csv"))
        test_df_x.to_csv(os.path.join(building_dir, "test.csv"))

        print(f"{name_building} processed and saved ✅")
    
    except Exception as e:
        print(f"Error processing {name_building}: {e}")
        continue
    
    # break
